# 3. HEB structures end to end

High-entropy diborides (HEB) like (Hf,Zr,Ti,Ta,Nb)B₂ have an AlB₂-type
structure — a chemically disordered metal sublattice (M) and a chemically
fixed boron sublattice (the **spectator**). SCRAPS only optimizes the M
sublattice; pairs involving boron are excluded from the fitness.

This notebook builds the canonical 5-metal 150-atom test case used in our
benchmarks.


## Build the AlB₂ primitive cell

3-atom basis: one M position at (0, 0, 0), two B positions at the honeycomb
sites. We give them placeholder species; the actual species are assigned via
`ELEMENT` in INPUT.


In [1]:
import sys
from pathlib import Path
REPO = Path.cwd().parent
sys.path.insert(0, str(REPO / "tools"))

import numpy as np
from ase import Atoms
from scraps_ase import run_sqs_from_atoms

# AlB2-type cell with hexagonal lattice and c/a = 1.1
a = 3.10  # rough HEB average lattice parameter (Å)
c = 1.1 * a
cell = [
    [a, 0, 0],
    [-0.5 * a, np.sqrt(3) / 2 * a, 0],
    [0, 0, c],
]
primitive = Atoms(
    symbols=["X"] * 3,
    scaled_positions=[
        [0.0, 0.0, 0.0],          # M (basis 0)
        [1/3, 2/3, 0.5],          # B (basis 1)
        [2/3, 1/3, 0.5],          # B (basis 2)
    ],
    cell=cell,
    pbc=True,
)
print(primitive)


Atoms(symbols='X3', pbc=True, cell=[[3.1, 0.0, 0.0], [-1.55, 2.68467875173176, 0.0], [0.0, 0.0, 3.4100000000000006]])


## Find the right cell size with `suggest_cell.py`

For a 5-metal equiatomic target on the M sublattice with ~150 total atoms,
the closest feasible cell is `Cell_DIM 5 5 2 = 150 atoms` (50 M + 100 B).
We pass `--spectator-basis-species "1:5,2:5"` so the feasibility math uses
M-only coordination.


In [2]:
import subprocess
work = Path.cwd() / "nb3_heb"
work.mkdir(exist_ok=True)
primitive.write(str(work / "POSCAR_primitive.vasp"))

suggest = REPO / "tools" / "suggest_cell.py"
result = subprocess.run(
    [sys.executable, str(suggest),
     str(work / "POSCAR_primitive.vasp"),
     "--composition", "0.2,0.2,0.2,0.2,0.2",
     "--target-atoms", "150",
     "--shells", "2",
     "--spectator-basis-species", "1:5,2:5",
     "--non-cubic"],
    capture_output=True, text=True,
)
print(result.stdout)


Composition fractions: [0.2, 0.2, 0.2, 0.2, 0.2]
Target atoms: 150
Weighted shells: 2 (weights = [1.0, 1.0])
Primitive basis atoms: 3

=== FEASIBLE cells (exact-SQS achievable, fitness can reach 0) ===
  Cell_DIM 2 5 5  ->  N=150 (+0 from target),  ELEMENT 10 10 10 10 10 100  [non-cubic]
  Cell_DIM 3 4 4  ->  N=144 (-6 from target),  ELEMENT 10 10 10 9 9 96  [non-cubic]
  Cell_DIM 2 4 6  ->  N=144 (-6 from target),  ELEMENT 10 10 10 9 9 96  [non-cubic]
  Cell_DIM 2 3 8  ->  N=144 (-6 from target),  ELEMENT 10 10 10 9 9 96  [non-cubic]
  Cell_DIM 3 3 6  ->  N=162 (+12 from target),  ELEMENT 11 11 11 11 10 108  [non-cubic]

=== INFEASIBLE cells (fitness floor > 0, plateau guaranteed) ===
  (none in search range; all configurations exact-SQS achievable)



## Run SCRAPS

Pass `spectator_basis_species=[(1, 5), (2, 5)]` to declare basis positions 1
and 2 as boron (species index 5 in `ELEMENT`). `species` lists the 6 element
symbols in `ELEMENT` order; the last must match the spectator species index.

**Important subtlety for HEB:** SCRAPS' `MAXSHELLNUM` counts *geometric*
shells over all atoms. In AlB₂ the first two shells are B-B and M-B
distances — they trivially contribute zero to the fitness because the
M-only filter excludes them. The first shell that actually carries M-M
pairs is **shell 2** (in-plane M-M at distance `a`), so we need at least
`max_shellnum=4` to optimize the first few M-M shells. With
`weights=[0, 0, 1, 1]` we explicitly ignore the B-only shells and put all
weight on the M-M ones.


In [3]:
heb = run_sqs_from_atoms(
    primitive,
    species=["Hf", "Zr", "Ti", "Ta", "Nb", "B"],
    element_counts=[10, 10, 10, 10, 10, 100],
    cell_dim=(5, 5, 2),
    max_shellnum=4,
    weights=[0, 0, 1, 1],     # ignore the B-B / M-B shells, weight M-M
    spectator_basis_species=[(1, 5), (2, 5)],
    rng_seed=42,
    work_dir=work / "run",
)
print("Atoms:", len(heb))
print("Symbol counts:", {s: list(heb.symbols).count(s)
                         for s in set(heb.symbols)})
print("Final fitness:", heb.info["scraps_final_fitness"])


Atoms: 150
Symbol counts: {'Hf': 10, 'Ta': 10, 'Ti': 10, 'Nb': 10, 'B': 100, 'Zr': 10}
Final fitness: 0.0


## Verify with spectator-aware `verify_sro.py`

`--spectator-species B` tells the verifier to compute the α matrix on the M
sublattice only — boron pairs are excluded. Note: SCRAPS' `MAXSHELLNUM` and
`verify_sro --shells` count *geometric* shells over all atoms, so in AlB₂ the
first couple of shells are B-B and M-B distances; the M-M shells start at
shell 3-4. We pass `--shells 6` here to actually see the M-M α matrix
populate.


In [4]:
verify = REPO / "tools" / "verify_sro.py"
work_dir = heb.info["scraps_work_dir"]
result = subprocess.run(
    [sys.executable, str(verify),
     str(Path(work_dir) / "SCRAPS.vasp"),
     "--species", "Hf,Zr,Ti,Ta,Nb,B",
     "--shells", "6",
     "--spectator-species", "B"],
    capture_output=True, text=True,
)
print(result.stdout)


Lattice: (15.5, 15.500005405160348, 6.82) (Angstrom), angles (90.0, 90.0, 119.99998846443923)
Atoms: 150 total, species ['Hf', 'Zr', 'Ti', 'Ta', 'Nb', 'B']
Detected shells: 3
  shell 0: r ∈ [1.7898, 1.7898], z = 2
  shell 1: r ∈ [2.4719, 2.4719], z = 8
  shell 2: r ∈ [3.1000, 3.1000], z = 6

=== Shell 2 (z = 6) ===
                Hf       Zr       Ti       Ta       Nb        B
    Hf  +0.0000 +0.0000 +0.0000 +0.0000 +0.0000    +nan 
    Zr  +0.0000 +0.0000 +0.0000 +0.0000 +0.0000    +nan 
    Ti  +0.0000 +0.0000 +0.0000 +0.0000 +0.0000    +nan 
    Ta  +0.0000 +0.0000 +0.0000 +0.0000 +0.0000    +nan 
    Nb  +0.0000 +0.0000 +0.0000 +0.0000 +0.0000    +nan 
     B     +nan    +nan    +nan    +nan    +nan    +nan 




Same recipe applies for any one-spectator HEB-class structure (other
borides, carbides like the (Hf,Zr,Ti,Ta,Nb)C high-entropy carbides, or
fluorite-like high-entropy oxides where one cation sublattice is the
designed one and the anion sublattice is the spectator).

For truly multi-sublattice cases where multiple sublattices have independent
designed compositions (σ-phase, Heusler with independent X₂/Y/Z compositions),
SCRAPS aborts with a guidance message pointing to ATAT's `mcsqs`. See
`docs/mcsqs_comparison.md` for the comparison.
